# LFW 01. ArcFace embedding extraction

목표: manifest의 얼굴 이미지에서 정규화된 ArcFace 512D 임베딩을 추출해 PostgreSQL 원본 임베딩 테이블에 저장합니다. 현재 파일, 처리량, 진행률, 처리 속도, ETA, 성공·실패·건너뜀 수를 30초 heartbeat와 100개 단위 checkpoint에서 표시합니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 00의 config hash와 `RUN_DIR/run_manifest.json`이 일치할 때만 재개하십시오. 중단되면 마지막 100개 DB checkpoint까지는 보존되며, 같은 run의 저장된 행은 자동으로 건너뜁니다. 00 입력 또는 실행 provider가 바뀌었으면 00부터 새 run을 만드십시오. `COMPLETED` run은 재개하지 않습니다.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
LIMIT = None  # Set a small integer for a recorded smoke run; use None for the full manifest.
USE_CUDA = True  # GTX 1080 Ti CUDA execution; set False only for an intentional CPU baseline.
PROGRESS_EVERY = 100
COMMIT_EVERY = 100
PROGRESS = ProgressReporter('01 ArcFace extraction', heartbeat_seconds=30)
print(f'Using CUDA: {USE_CUDA}; progress/checkpoint interval: {PROGRESS_EVERY}')


Using CUDA: True; progress/checkpoint interval: 100


## Plan

- Attach to the frozen run and reject completed/mismatched state.
- Decode each image, detect the face, and extract a normalized ArcFace 512D vector.
- Show the current file and live progress every 30 seconds.
- Commit every 100 images so an interrupted run can resume from its last checkpoint.
- Store a final ledger and failure report with phase checksums.

In [2]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable; start again from notebook 00.')
    return run, manifest

preflight = {
    'execute_stage': EXECUTE_STAGE,
    'run_dir_resolved': str(RUN_DIR),
    'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file()),
    'cuda_requested': USE_CUDA,
    'limit': LIMIT,
}
preflight


{'execute_stage': True,
 'run_dir_resolved': 'D:\\ronbun\\runs\\lfw\\2026\\07\\15\\20260715-R001-c305da87_thesis3_lfw_face_search_v1',
 'run_manifest_exists': True,
 'cuda_requested': True,
 'limit': None}

## Execute and record

출력의 `processed`, `percent`, `current_file`, `images_per_second`, `eta`, `inserted`, `skipped`, `failed`를 확인합니다. `DB checkpoint committed`가 출력된 구간은 커널 중단 후에도 보존됩니다.

DB 비밀번호는 노트북에 쓰지 않습니다. `RONBUN_DB_PASSWORD` 또는 Git에서 제외된 `configs/database.local.yaml`을 사용합니다.

In [3]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    from time import perf_counter

    import cv2
    import onnxruntime as ort
    import pandas as pd
    from research.compression import ORIGIN_512
    from research.database import VectorRepository, create_database_engine, init_database, load_database_settings, session_scope
    from research.embeddings import ArcFaceFeatureExtractor, FaceAnalysisSettings
    from research.runtime.hashing import sha256_file
    from research.runtime.redaction import redact

    def duration_text(seconds: float | None) -> str | None:
        if seconds is None:
            return None
        total = max(0, int(round(seconds)))
        hours, remainder = divmod(total, 3600)
        minutes, seconds = divmod(remainder, 60)
        return f'{hours:d}h {minutes:02d}m {seconds:02d}s' if hours else f'{minutes:d}m {seconds:02d}s'

    run, run_manifest = attach_run(RUN_DIR)
    run.verify_inputs()
    config = run_manifest['config']
    manifest_path = PROJECT_ROOT / config['dataset']['manifest_path']
    rows = pd.read_csv(manifest_path)
    if LIMIT is not None:
        rows = rows.head(int(LIMIT))
    total_rows = int(len(rows))
    if total_rows == 0:
        raise ValueError('No manifest rows were selected for extraction.')
    available_providers = tuple(ort.get_available_providers())
    if USE_CUDA and 'CUDAExecutionProvider' not in available_providers:
        raise RuntimeError(
            'USE_CUDA=True but CUDAExecutionProvider is unavailable: '
            f'{available_providers}. Restart the kernel after installing onnxruntime-gpu.'
        )
    providers = ('CUDAExecutionProvider', 'CPUExecutionProvider') if USE_CUDA else ('CPUExecutionProvider',)
    extractor = ArcFaceFeatureExtractor(FaceAnalysisSettings(providers=providers))
    engine = create_database_engine(load_database_settings())
    init_database(engine)
    counts = {'processed': 0, 'inserted': 0, 'skipped': 0, 'failed': 0}
    failures = []
    ledger = []
    progress_state = {
        'completed': 0,
        'total': total_rows,
        'current_file': None,
        'started_at': perf_counter(),
        'last_checkpoint': 0,
    }

    def progress_details() -> dict[str, object]:
        completed = int(progress_state['completed'])
        elapsed = max(perf_counter() - float(progress_state['started_at']), 1e-9)
        rate = completed / elapsed if completed else 0.0
        remaining = total_rows - completed
        eta_seconds = remaining / rate if rate > 0 else None
        return {
            'processed': f'{completed}/{total_rows}',
            'percent': f'{completed / total_rows * 100:.1f}%',
            'current_file': progress_state['current_file'],
            'images_per_second': f'{rate:.2f}',
            'eta': duration_text(eta_seconds),
            'inserted': counts['inserted'],
            'skipped': counts['skipped'],
            'failed': counts['failed'],
            'last_checkpoint': progress_state['last_checkpoint'],
        }

    PROGRESS.emit(
        'workload ready',
        total=total_rows,
        providers_requested=list(providers),
        providers_available=list(available_providers),
        progress_every=PROGRESS_EVERY,
        commit_every=COMMIT_EVERY,
    )
    with run.phase('01_arcface_embedding_extraction') as phase:
        with PROGRESS.step(
            'manifest image extraction',
            expected='CPU/GPU provider와 이미지 수에 따라 달라짐',
            heartbeat_details=progress_details,
        ):
            with session_scope(engine) as session:
                repository = VectorRepository(session)
                for position, row in enumerate(rows.to_dict(orient='records'), start=1):
                    image_path = Path(str(row['image_path']))
                    image_path = image_path if image_path.is_absolute() else PROJECT_ROOT / image_path
                    try:
                        display_path = str(image_path.relative_to(PROJECT_ROOT))
                    except ValueError:
                        display_path = str(image_path)
                    progress_state['current_file'] = display_path
                    content_sha256 = None
                    counts['processed'] += 1
                    try:
                        if not image_path.is_file():
                            raise FileNotFoundError(image_path)
                        content_sha256 = sha256_file(image_path)
                        file_size_bytes = image_path.stat().st_size
                        with session.begin_nested():
                            image = repository.add_image(
                                str(image_path), label=str(row['identity_id']),
                                content_sha256=content_sha256, file_size_bytes=file_size_bytes,
                            )
                            existing = repository.get_embeddings_512(
                                image_id=image.id, vector_type=ORIGIN_512, run_uid=run.run_id
                            )
                            if existing:
                                counts['skipped'] += 1
                                ledger.append({'image_path': str(image_path), 'content_sha256': content_sha256,
                                               'status': 'skipped_existing', 'embedding_id': existing[0].id})
                            else:
                                pixels = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
                                extracted = extractor.extract_with_metadata(pixels)
                                embedding_record = repository.add_embedding_512(
                                    image.id, ORIGIN_512,
                                    {
                                        'run_id': run.run_id,
                                        'model': extractor.settings.model_name,
                                        'l2_normalized': True,
                                        'providers_requested': list(providers),
                                    },
                                    extracted.embedding,
                                    log=json.dumps({'face_count': extracted.face_count, 'bbox': extracted.bbox, 'detection_score': extracted.detection_score}),
                                    run_uid=run.run_id,
                                )
                                counts['inserted'] += 1
                                ledger.append({'image_path': str(image_path), 'content_sha256': content_sha256,
                                               'status': 'inserted', 'embedding_id': embedding_record.id})
                    except Exception as exc:
                        counts['failed'] += 1
                        failure = redact({'image_path': str(image_path), 'content_sha256': content_sha256,
                                          'error_type': type(exc).__name__, 'message': str(exc)})
                        failures.append(failure)
                        ledger.append(redact({'image_path': str(image_path), 'content_sha256': content_sha256,
                                              'status': 'failed', 'embedding_id': None,
                                              'error_type': type(exc).__name__, 'message': str(exc)}))
                    finally:
                        progress_state['completed'] = position

                    if position % COMMIT_EVERY == 0:
                        repository.commit()
                        progress_state['last_checkpoint'] = position
                        PROGRESS.emit('DB checkpoint committed', **progress_details())
                    elif position == 1 or position % PROGRESS_EVERY == 0 or position == total_rows:
                        PROGRESS.emit('extraction progress', **progress_details())

                repository.commit()
                progress_state['last_checkpoint'] = total_rows
        report_path = phase.attempt_dir / f'extraction_summary_A{phase.attempt:03d}.json'
        ledger_path = phase.attempt_dir / f'extraction_ledger_A{phase.attempt:03d}.csv'
        report_payload = redact({
            'counts': counts,
            'failures': failures,
            'providers_requested': list(providers),
            'providers_available': list(available_providers),
            'progress_every': PROGRESS_EVERY,
            'commit_every': COMMIT_EVERY,
        })
        report_path.write_text(json.dumps(report_payload, ensure_ascii=False, indent=2), encoding='utf-8')
        pd.DataFrame.from_records(redact(ledger)).to_csv(ledger_path, index=False)
        phase.publish_artifact(report_path)
        phase.publish_artifact(ledger_path)
        phase.record_counts(**counts)
        phase.record(
            'extraction_runtime',
            providers_requested=list(providers),
            providers_available=list(available_providers),
            progress_every=PROGRESS_EVERY,
            commit_every=COMMIT_EVERY,
        )
    PROGRESS.emit('01 completed', **progress_details())
    result = {'status': 'completed', 'run_id': run.run_id, **counts}
result

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\Administrator/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionP

c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


[13:05:59] 01 ArcFace extraction | extraction progress | elapsed=13s | processed=1/13233 percent=0.0% current_file=data\raw\LFW\lfw-deepfunneled\lfw-deepfunneled\Aaron_Pena\Aaron_Pena_0001.jpg images_per_second=0.15 eta=24h 14m 20s inserted=1 skipped=0 failed=0 last_checkpoint=0
[13:06:05] 01 ArcFace extraction | DB checkpoint committed | elapsed=19s | processed=100/13233 percent=0.8% current_file=data\raw\LFW\lfw-deepfunneled\lfw-deepfunneled\Alma_Powell\Alma_Powell_0001.jpg images_per_second=7.82 eta=27m 58s inserted=100 skipped=0 failed=0 last_checkpoint=100
[13:06:12] 01 ArcFace extraction | DB checkpoint committed | elapsed=26s | processed=200/13233 percent=1.5% current_file=data\raw\LFW\lfw-deepfunneled\lfw-deepfunneled\Azmi_Bishara\Azmi_Bishara_0001.jpg images_per_second=10.33 eta=21m 02s inserted=200 skipped=0 failed=0 last_checkpoint=200
[13:06:18] 01 ArcFace extraction | DB checkpoint committed | elapsed=32s | processed=300/13233 percent=2.3% current_file=data\raw\LFW\lfw-dee

{'status': 'completed',
 'run_id': '20260715-R001-c305da87',
 'processed': 13233,
 'inserted': 13195,
 'skipped': 0,
 'failed': 38}

## Next step

마지막 출력에서 `processed == total`, `last_checkpoint == total`, `failed == 0`인지 확인한 뒤 02로 이동합니다. 실패가 있으면 `extraction_summary`와 `extraction_ledger`에서 현재 파일과 오류를 확인합니다. 중단되었다면 Kernel Restart 후 01 전체를 다시 실행하면 마지막 DB checkpoint까지 저장된 행을 건너뜁니다.